In [1]:
import pandas as pd
import numpy as np

1. Establish a connection between Python and the Sakila database.

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Table, MetaData, text


# relative path for the sql database
sqlfile = '../../inclass-data/sqlite-sakila.db'
engine = create_engine('sqlite:///'+sqlfile)        # turn diagnostic output of SQL connection off to reduce clutter



2. Write a Python function called `rentals_month` that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:

	- `engine`: an object representing the database connection engine to be used to establish a connection to the Sakila database.
	- `month`: an integer representing the month for which rental data is to be retrieved.
	- `year`: an integer representing the year for which rental data is to be retrieved.

	The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.

In [14]:
def rentals_month(engine, month, year):
    query = text("""
    SELECT r.* FROM rental AS r
    WHERE date(rental_date) LIKE :yearmonth
    """)
    with engine.connect() as conn:
            return pd.DataFrame(conn.execute(query, {'yearmonth': f"{year}-{str(month).zfill(2)}%"}).fetchall())

In [36]:
rentals_month(engine,7,2005)

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,3470,2005-07-05 22:49:24.000,883,565,2005-07-07 19:36:24.000,1,2021-03-06 15:54:09
1,3471,2005-07-05 22:51:44.000,1724,242,2005-07-13 01:38:44.000,2,2021-03-06 15:54:09
2,3472,2005-07-05 22:56:33.000,841,37,2005-07-13 17:18:33.000,2,2021-03-06 15:54:09
3,3473,2005-07-05 22:57:34.000,2735,60,2005-07-12 23:53:34.000,1,2021-03-06 15:54:09
4,3474,2005-07-05 22:59:53.000,97,594,2005-07-08 20:32:53.000,1,2021-03-06 15:54:09
...,...,...,...,...,...,...,...
6704,10176,2005-07-31 23:40:35.000,1181,19,2005-08-09 00:46:35.000,2,2021-03-06 15:55:04
6705,10177,2005-07-31 23:42:33.000,2242,279,2005-08-03 01:30:33.000,2,2021-03-06 15:55:04
6706,10178,2005-07-31 23:43:04.000,1582,491,2005-08-03 00:43:04.000,1,2021-03-06 15:55:04
6707,10179,2005-07-31 23:49:54.000,2136,131,2005-08-01 20:46:54.000,2,2021-03-06 15:55:04


3. Develop a Python function called `rental_count_month` that takes the DataFrame provided by `rentals_month` as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year. 

	The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".


	*Hint: Consider making use of pandas [groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)*



In [28]:
def rental_count_month(df, month, year):
    return df.groupby('customer_id')['rental_id'].count().reset_index(name=f"rentals_{str(month).zfill(2)}_{str(year)}")

In [ ]:
rental_count_month(rentals_month(engine,7,2005),7,2005)

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3
...,...,...
515,594,4
516,595,1
517,596,6
518,597,2


In [37]:
# it might be safer for data consistency to collect the data and do the grouping and column-naming in the same step
# this avoids inconsistencies for the month and year arguments

def rental_count_month_onestep(engine, month, year):
    ymstring = f"{year}-{str(month).zfill(2)}"
    query = text("""
    SELECT r.* FROM rental AS r
    WHERE date(rental_date) LIKE :yearmonth
    """)
    with engine.connect() as conn:
            df = pd.DataFrame(conn.execute(query, {'yearmonth': ymstring+'%'}).fetchall())
        
    if df.empty:
        print("The search returned an empty dataframe")
        return None
    else:
        return df.groupby('customer_id')['rental_id'].count().reset_index(name=f"rentals_{ymstring}")


In [39]:
rental_count_month_onestep(engine,6,2005)

,customer_id,rentals_2005-06
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5
...,...,...
585,595,2
586,596,2
587,597,3
588,598,1


4. Create a Python function called `compare_rentals` that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. 
The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [61]:
def compare_rentals(df1,df2):
    newdf = df1.merge(df2,on='customer_id',how='outer',suffixes=('_1', '_2'))
    newdf.fillna(0, inplace = True)                                         # ensure NaN values for customers with 0 rentals for one of the compared months are set to 0
    yearmon = [c for c in newdf.columns if 'rentals_' in c]
    if len(yearmon) != 2:
        print(f'Something went wrong, expecting two columns with rental counts, got {len(yearmon)}')
        print(yearmon)
        return newdf
    else:
        newdf['difference'] = newdf[yearmon[0]] - newdf[yearmon[1]]
        return newdf

In [60]:
compare_rentals(rental_count_month_onestep(engine,5,2005),rental_count_month_onestep(engine,6,2005))

,customer_id,rentals_2005-05,rentals_2005-06,difference
0,1,2.0,7.0,-5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,-2.0
3,4,0.0,6.0,-6.0
4,5,3.0,5.0,-2.0
...,...,...,...,...
593,595,1.0,2.0,-1.0
594,596,6.0,2.0,4.0
595,597,2.0,3.0,-1.0
596,598,0.0,1.0,-1.0
